# Rôle 1 : modèles de rentabilité et de risque

1. Préparation : jeux, anomalies, mesures
2. Modèles de référence (tâche 5) : `DummyRegressor`, `LinearRegression`, `DummyClassifier`

Règles suivies dans tout le notebook :

- les modèles sont appris sur le **jeu d'entraînement** (1 050 projets) et comparés sur le **jeu de validation** (225 projets) ;
- le prétraitement (`supervise.creer_pretraitement`) est placé dans un `Pipeline` avec le modèle : il n'apprend que sur l'entraînement ;
- le **jeu de test n'est pas utilisé ici** : il sera ouvert une seule fois, à la fin, pour le modèle choisi (tâche 8).

Rappel : `rentabilite` et `risque` sont **simulées** (voir `data/README.md`) ; le bruit de la simulation empêche tout modèle d'être parfait. Le contrôle du script de préparation donne un R² d'environ 0,55 et 58 % de bonnes classes de risque avec un modèle linéaire.

In [1]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline

RACINE = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RACINE))
from src import supervise

logging.basicConfig(level=logging.INFO, format="%(message)s")
GRAINE = 42
pd.set_option("display.width", 120)

## 1. Préparation

In [2]:
jeux = supervise.charger_jeux()
entrainement, validation = jeux["entrainement"], jeux["validation"]

anomalies = pd.read_csv(RACINE / "data" / "processed" / "anomalies_injectees.csv")
ids_anomalies = set(anomalies["id_projet"])
ids_cibles_fabriquees = set(anomalies.loc[anomalies["type_anomalie"] == "rentabilite_extreme", "id_projet"])

pd.DataFrame({
    nom: [len(jeu), jeu["id_projet"].isin(ids_anomalies).sum(), jeu["id_projet"].isin(ids_cibles_fabriquees).sum()]
    for nom, jeu in jeux.items()
}, index=["projets", "anomalies injectées", "dont rentabilité fabriquée"])

Jeux chargés : {'entrainement': 1050, 'validation': 225, 'test': 225}


,entrainement,validation,test
projets,1050,225,225
anomalies injectées,17,5,3
dont rentabilité fabriquée,6,2,2


Les anomalies sont de deux natures très différentes :

- **coût ou durée modifiés** (`cout_aberrant`, `duree_incoherente`) : l'**entrée** est fausse, mais la cible est celle du vrai projet ;
- **rentabilité fabriquée** (`rentabilite_extreme`) : la **cible** elle-même est remplacée par une valeur au hasard entre 35 et 60 %, positive ou négative. Aucun modèle ne peut la prédire, puisqu'elle ne dépend de rien.

Le jeu de validation contient 2 rentabilités fabriquées (+38,1 % et −56,0 %). On verra qu'à elles seules elles écrasent les mesures de régression. On calcule donc les mesures sur deux versions de la validation :

- **validation complète** (225 projets), telle qu'elle arrive ;
- **validation sans cibles fabriquées** (223 projets) : la mesure de ce que le modèle sait vraiment prédire.

Le risque n'est pas touché : il a été calculé avant l'injection des anomalies.

Le fichier des anomalies ne sert **jamais d'entrée** au modèle : seulement à choisir quels projets garder pour l'apprentissage et à lire les résultats.

In [3]:
validation_sans_fabriquees = validation[~validation["id_projet"].isin(ids_cibles_fabriquees)]
jeux_evaluation = {"validation complète": validation, "validation sans cibles fabriquées": validation_sans_fabriquees}


def pipeline(modele) -> Pipeline:
    """Prétraitement commun suivi du modèle : tout s'ajuste sur les données passées à fit."""
    return Pipeline([("pretraitement", supervise.creer_pretraitement()), ("modele", modele)])


def mesures_regression(y_vrai, y_predit) -> dict:
    """RMSE, MAE (en points de rentabilité) et R²."""
    return {"RMSE": root_mean_squared_error(y_vrai, y_predit),
            "MAE": mean_absolute_error(y_vrai, y_predit),
            "R2": r2_score(y_vrai, y_predit)}


def mesures_classification(y_vrai, y_predit) -> dict:
    """Exactitude et F1 macro (moyenne des F1 des trois classes, chacune avec le même poids)."""
    return {"exactitude": accuracy_score(y_vrai, y_predit),
            "F1 macro": f1_score(y_vrai, y_predit, average="macro")}

**Les mesures.**

- **RMSE** (racine de l'erreur quadratique moyenne) : l'erreur typique, en points de rentabilité. Elle élève les erreurs au carré, donc punit fort les grosses erreurs.
- **MAE** (erreur absolue moyenne) : l'erreur moyenne en points, moins sensible aux valeurs extrêmes.
- **R²** : part de la variance de la rentabilité expliquée par le modèle. 0 = pas mieux que prédire la moyenne ; 1 = parfait ; négatif = pire que la moyenne.
- **Exactitude** : part de projets bien classés.
- **F1 macro** : pour chaque classe, le F1 combine précision et rappel ; on fait la moyenne des trois classes sans pondérer par leur taille. Un modèle qui ignore la classe rare `eleve` est donc pénalisé, alors que son exactitude peut rester correcte.

## 2. Modèles de référence (tâche 5)

Un modèle de référence donne le niveau à battre. Sans lui, un R² ou une exactitude ne veulent rien dire.

- `DummyRegressor` : prédit toujours la **moyenne** de la rentabilité d'entraînement ;
- `LinearRegression` : régression linéaire par moindres carrés, sans régularisation ;
- `DummyClassifier` : répond toujours la **classe la plus fréquente** (`most_frequent`), ou tire une classe au hasard selon les proportions d'entraînement (`stratified`).

### 2.1 Régression de la rentabilité

On apprend chaque modèle sur trois versions de l'entraînement, pour mesurer l'effet des anomalies :

- **tout l'entraînement** (1 050 projets) ;
- **sans les rentabilités fabriquées** (1 044 projets) ;
- **sans aucune anomalie connue** (1 033 projets).

In [4]:
versions_entrainement = {
    "tout l'entraînement": entrainement,
    "sans rentabilités fabriquées": entrainement[~entrainement["id_projet"].isin(ids_cibles_fabriquees)],
    "sans anomalies connues": entrainement[~entrainement["id_projet"].isin(ids_anomalies)],
}
modeles_reference = {"DummyRegressor (moyenne)": DummyRegressor(strategy="mean"),
                     "LinearRegression": LinearRegression()}

lignes = []
for nom_entrainement, donnees in versions_entrainement.items():
    for nom_modele, modele in modeles_reference.items():
        ajuste = pipeline(modele).fit(donnees[supervise.VARIABLES], donnees[supervise.CIBLE_REGRESSION])
        ligne = {"modèle": nom_modele, "entraînement": nom_entrainement, "projets appris": len(donnees)}
        ligne["R2 entraînement"] = r2_score(donnees[supervise.CIBLE_REGRESSION],
                                            ajuste.predict(donnees[supervise.VARIABLES]))
        for nom_evaluation, evaluation in jeux_evaluation.items():
            predit = ajuste.predict(evaluation[supervise.VARIABLES])
            for mesure, valeur in mesures_regression(evaluation[supervise.CIBLE_REGRESSION], predit).items():
                ligne[f"{mesure} ({nom_evaluation})"] = valeur
        lignes.append(ligne)

resultats_reference = pd.DataFrame(lignes)
resultats_reference.round(3)

,modèle,entraînement,projets appris,R2 entraînement,RMSE (validation complète),MAE (validation complète),R2 (validation complète),RMSE (validation sans cibles fabriquées),MAE (validation sans cibles fabriquées),R2 (validation sans cibles fabriquées)
0,DummyRegressor (moyenne),tout l'entraînement,1050,0.000,5.496,2.660,-0.021,2.803,2.262,-0.053
1,LinearRegression,tout l'entraînement,1050,0.405,5.286,2.210,0.056,2.413,1.816,0.219
2,DummyRegressor (moyenne),sans rentabilités fabriquées,1044,0.000,5.478,2.639,-0.014,2.775,2.240,-0.033
3,LinearRegression,sans rentabilités fabriquées,1044,0.665,5.230,2.035,0.076,2.142,1.632,0.385
4,DummyRegressor (moyenne),sans anomalies connues,1033,0.000,5.478,2.638,-0.014,2.774,2.240,-0.032
5,LinearRegression,sans anomalies connues,1033,0.668,5.226,2.031,0.077,2.134,1.629,0.389


In [5]:
colonnes = ["modèle", "entraînement", "R2 entraînement",
            "RMSE (validation sans cibles fabriquées)", "MAE (validation sans cibles fabriquées)",
            "R2 (validation sans cibles fabriquées)", "R2 (validation complète)"]
resultats_reference[colonnes].round(3)

,modèle,entraînement,R2 entraînement,RMSE (validation sans cibles fabriquées),MAE (validation sans cibles fabriquées),R2 (validation sans cibles fabriquées),R2 (validation complète)
0,DummyRegressor (moyenne),tout l'entraînement,0.000,2.803,2.262,-0.053,-0.021
1,LinearRegression,tout l'entraînement,0.405,2.413,1.816,0.219,0.056
2,DummyRegressor (moyenne),sans rentabilités fabriquées,0.000,2.775,2.240,-0.033,-0.014
3,LinearRegression,sans rentabilités fabriquées,0.665,2.142,1.632,0.385,0.076
4,DummyRegressor (moyenne),sans anomalies connues,0.000,2.774,2.240,-0.032,-0.014
5,LinearRegression,sans anomalies connues,0.668,2.134,1.629,0.389,0.077


Ce que montre le tableau :

1. **Le modèle « moyenne » a un R² proche de 0**, par construction : c'est le plancher. Son erreur typique est d'environ 2,8 points sur la validation sans cibles fabriquées.
2. **Les 2 rentabilités fabriquées de la validation écrasent les mesures** : avec elles, la RMSE fait plus que doubler (de 2,1 à 5,2 points) et le R² de la régression linéaire tombe sous 0,1. Ces deux projets représentent 1 % de la validation mais l'essentiel de l'erreur au carré. Comparer des modèles sur cette version reviendrait surtout à comparer leurs erreurs sur deux valeurs tirées au hasard.
3. **Les 6 rentabilités fabriquées de l'entraînement faussent l'apprentissage** : les retirer fait passer le R² de validation de la régression linéaire d'environ 0,22 à 0,39. Retirer en plus les 11 projets à coût ou durée modifiés ne change presque rien : le log10 du coût atténue déjà les coûts multipliés par 10 à 50.
4. **La régression linéaire apprend trop les données d'entraînement** : R² de 0,67 sur l'entraînement contre 0,39 en validation. Avec 175 colonnes pour un millier de projets, dont 158 pays souvent représentés par un ou deux projets, elle donne à chaque pays rare un coefficient qui colle au bruit de ces quelques projets. C'est exactement ce que la régularisation (Ridge, Lasso, ElasticNet, tâche 6) doit corriger, et ce qui explique qu'on reste sous le plafond d'environ 0,55 annoncé par le script de préparation.

### 2.2 Classification du risque

In [6]:
lignes = []
for strategie in ["most_frequent", "stratified"]:
    ajuste = pipeline(DummyClassifier(strategy=strategie, random_state=GRAINE)).fit(
        entrainement[supervise.VARIABLES], entrainement[supervise.CIBLE_CLASSIFICATION])
    predit = ajuste.predict(validation[supervise.VARIABLES])
    lignes.append({"modèle": f"DummyClassifier ({strategie})",
                   **mesures_classification(validation[supervise.CIBLE_CLASSIFICATION], predit)})
resultats_reference_risque = pd.DataFrame(lignes)
resultats_reference_risque.round(3)

,modèle,exactitude,F1 macro
0,DummyClassifier (most_frequent),0.400,0.190
1,DummyClassifier (stratified),0.356,0.333


- **Toujours la classe la plus fréquente** : 40 % d'exactitude (la classe `faible`, à égalité avec `moyen`, représente 40 % des projets), mais un F1 macro de 0,19, car deux classes sur trois ne sont jamais prédites.
- **Tirage au hasard selon les proportions** : exactitude d'environ 36 % et F1 macro d'environ 0,33 ; il prédit toutes les classes, mais au hasard.

Ce sont les niveaux à battre : un classifieur utile doit dépasser **40 % d'exactitude** et un **F1 macro de 0,33**. Le contrôle du script de préparation annonce environ 58 % pour un modèle linéaire.

Pour le risque, les anomalies n'ont pas d'effet sur les étiquettes (calculées avant l'injection) ; on garde tous les projets de validation.

### 2.3 Bilan des modèles de référence

| Tâche | Référence | Validation |
|---|---|---|
| Rentabilité | moyenne d'entraînement | R² ≈ 0, RMSE ≈ 2,8 points |
| Rentabilité | régression linéaire, sans rentabilités fabriquées | R² ≈ 0,39, RMSE ≈ 2,1 points |
| Risque | classe la plus fréquente | exactitude 40 %, F1 macro 0,19 |
| Risque | tirage selon les proportions | exactitude ≈ 36 %, F1 macro ≈ 0,33 |

Choix pour la suite :

- les modèles de rentabilité sont appris **sans les rentabilités fabriquées connues** de l'entraînement ;
- ils sont comparés d'abord sur la **validation sans cibles fabriquées**, et on donne aussi la mesure sur la validation complète pour être transparent.